In [1]:
import pandas as pd
import numpy as np

print('the jupyter works')

the jupyter works


In [2]:
from huggingface_hub import login

login()

/home/g_husky/code/DATA_PRIVACY/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from huggingface_hub import HfFileSystem
import json
from datasets import Dataset

fs = HfFileSystem()
REPO = "datasets/iamgroot42/mimir"

def load_mimir(config: str, split: str | None = "ngram_7_0.2") -> Dataset:
    suffix = f"_{split}" if split and split != "none" else ""
    member = fs.glob(f"{REPO}/cache_100_200_1000_512/train/{config}{suffix}.jsonl")
    nonmember = fs.glob(f"{REPO}/cache_100_200_1000_512/test/{config}{suffix}.jsonl")
    if not member or not nonmember:
        raise FileNotFoundError(f"No file for config={config!r} split={split!r}")

    inputs, labels = [], []
    with fs.open(member[0]) as fm, fs.open(nonmember[0]) as fn:
        for m_line, n_line in zip(fm, fn):
            inputs.append(json.loads(m_line)); labels.append(1)   # member
            inputs.append(json.loads(n_line)); labels.append(0)   # nonmember

    return Dataset.from_dict({"input": inputs, "label": labels})

dataset = load_mimir("arxiv", "ngram_7_0.2")
print(dataset.shape)

(1000, 2)


In [4]:
# load dataset for mimir is deprecated

from datasets import load_dataset

dataset = load_dataset(
    "iamgroot42/mimir", 
    "arxiv", 
    split="ngram_7_0.2", 
    trust_remote_code=True  # Fixed typo: 'trust_remote_code'
)
print(dataset)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'iamgroot42/mimir' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


RuntimeError: Dataset scripts are no longer supported, but found mimir.py

In [5]:
# this was working, but the moment I did uv add huggingface_hub, it stopped working.

import pandas as pd

splits = {'WikiMIA_length128': 'data/WikiMIA_length128-00000-of-00001-fff31bd5ed612836.parquet', 'WikiMIA_length256': 'data/WikiMIA_length256-00000-of-00001-e984cf40f6c5b556.parquet', 'WikiMIA_length32': 'data/WikiMIA_length32-00000-of-00001-6d31a92f6d59bcdc.parquet', 'WikiMIA_length64': 'data/WikiMIA_length64-00000-of-00001-c337a02056685c1a.parquet'}
df = pd.read_parquet("hf://datasets/swj0419/WikiMIA/" + splits["WikiMIA_length128"])

AttributeError: module 'fsspec' has no attribute 'url_to_fs'

try to load a subset of the pile

In [ ]:
# from datasets import load_dataset

# # Downloads only the first 100 rows of the training set
# small_pile = load_dataset("monology/pile-uncopyrighted", split="test[:100]")

# print(small_pile[0]["text"][:10])

In [ ]:
from datasets import load_dataset, Dataset

print("Loading Train Set (Members)...")
# 1. Get the members (Train set) from the uncopyrighted train repo
train_dataset = load_dataset(
    "monology/pile-uncopyrighted", 
    split="train",
    revision="refs/convert/parquet",
    streaming=True
)

print("Loading Test Set (Non-Members)...")
# 2. Get the non-members (Test set) from the dedicated test/val repo
# (No special revision needed here, it's on the main branch)
test_dataset = load_dataset(
    "monology/pile-test-val",
    split="test",  # You can also use "validation" here if you prefer
    streaming=True
)

# 3. Create the domain filter
def is_pubmed(example):
    return example["meta"]["pile_set_name"] == "PubMed Abstracts"

print("Filtering both streams for PubMed...")
pubmed_train_stream = train_dataset.filter(is_pubmed)
pubmed_test_stream = test_dataset.filter(is_pubmed)

# 4. Extract exactly what you need for the MIA
# (Takes a moment to scan over the network)
my_train_members = list(pubmed_train_stream.take(1000))
my_test_non_members = list(pubmed_test_stream.take(1000))

print(f"Success! Fetched {len(my_train_members)} Members and {len(my_test_non_members)} Non-Members.")

# 5. Save locally for your experiment
Dataset.from_list(my_train_members).to_parquet("mia_members_train.parquet")
Dataset.from_list(my_test_non_members).to_parquet("mia_non_members_test.parquet")

In [9]:
import zstandard as zstd
import requests
import io
import json
import pandas as pd

def stream_pubmed_test_set(max_rows=1000):
    # Direct download link to the exact test file
    url = "https://huggingface.co/datasets/monology/pile-test-val/resolve/main/test.jsonl.zst"
    
    print("Connecting to Hugging Face test server...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    # Set up the on-the-fly decompression stream
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(response.raw)
    text_stream = io.TextIOWrapper(stream_reader, encoding='utf-8')
    
    pubmed_data = []
    
    print("Scanning compressed stream for PubMed data (this will take a moment)...")
    for line in text_stream:
        row = json.loads(line)
        
        # Filter for our domain
        if row.get("meta", {}).get("pile_set_name") == "PubMed Abstracts":
            pubmed_data.append(row)
            
            # Stop once we have what we need
            if len(pubmed_data) >= max_rows:
                break
                
    return pubmed_data

# 1. Fetch exactly 1000 True Non-Members directly
my_test_non_members = stream_pubmed_test_set(1000)

print(f"Success! Fetched {len(my_test_non_members)} Non-Members.")

# 2. Save them straight to Parquet using pandas
df = pd.DataFrame(my_test_non_members)
df.to_parquet("mia_non_members_test.parquet")
print("Saved to mia_non_members_test.parquet!")

Connecting to Hugging Face test server...
Scanning compressed stream for PubMed data (this will take a moment)...
Success! Fetched 1000 Non-Members.
Saved to mia_non_members_test.parquet!


In [ ]:
df = pd.read_parquet("mia_non_members_test.parquet")

In [14]:
df.head()

,text,meta
0,One-year follow-up of patients treated for den...,{'pile_set_name': 'PubMed Abstracts'}
1,Optimizing medical practice using a computeriz...,{'pile_set_name': 'PubMed Abstracts'}
2,Antioxidant protection against acoustic trauma...,{'pile_set_name': 'PubMed Abstracts'}
3,Cross coupling reactions of organozinc iodides...,{'pile_set_name': 'PubMed Abstracts'}
4,Removable prosthodontic therapy and xerostomia...,{'pile_set_name': 'PubMed Abstracts'}


In [16]:
import pandas as pd
from datasets import load_dataset
import zstandard as zstd
import requests
import io
import json

# ==========================================
# CONFIGURATION
# ==========================================
TRAIN_ROWS_NEEDED = 100  # True Members
TEST_ROWS_NEEDED = 100   # True Non-Members
TARGET_DOMAIN = "PubMed Abstracts"

# ==========================================
# STEP 1: FETCH TRAIN SET (MEMBERS)
# Method: Hugging Face Datasets via Parquet
# ==========================================
print(f"--- Fetching {TRAIN_ROWS_NEEDED} Train Rows (Members) ---")

# 1. Load the streaming Parquet branch
train_dataset = load_dataset(
    "monology/pile-uncopyrighted", 
    split="train",
    revision="refs/convert/parquet",
    streaming=True
)

# 2. Filter and extract
def is_pubmed(example):
    return example["meta"]["pile_set_name"] == TARGET_DOMAIN

pubmed_train_stream = train_dataset.filter(is_pubmed)
train_members_list = list(pubmed_train_stream.take(TRAIN_ROWS_NEEDED))

# 3. Save to Parquet
df_train = pd.DataFrame(train_members_list)
df_train.to_parquet("mia_members_train.parquet")
print(f"Saved {len(df_train)} rows to mia_members_train.parquet\n")


# ==========================================
# STEP 2: FETCH TEST SET (NON-MEMBERS)
# Method: Direct zstandard streaming
# ==========================================
print(f"--- Fetching {TEST_ROWS_NEEDED} Test Rows (Non-Members) ---")

def stream_test_set(max_rows, domain):
    url = "https://huggingface.co/datasets/monology/pile-test-val/resolve/main/test.jsonl.zst"
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(response.raw)
    text_stream = io.TextIOWrapper(stream_reader, encoding='utf-8')
    
    extracted_data = []
    
    for line in text_stream:
        row = json.loads(line)
        if row.get("meta", {}).get("pile_set_name") == domain:
            extracted_data.append(row)
            if len(extracted_data) >= max_rows:
                break
                
    return extracted_data

# 1. Stream and extract
test_non_members_list = stream_test_set(TEST_ROWS_NEEDED, TARGET_DOMAIN)

# 2. Save to Parquet
df_test = pd.DataFrame(test_non_members_list)
df_test.to_parquet("mia_non_members_test.parquet")
print(f"Saved {len(df_test)} rows to mia_non_members_test.parquet\n")


# ==========================================
# STEP 3: VERIFICATION
# ==========================================
print("--- Final Verification ---")
check_train = pd.read_parquet("mia_members_train.parquet")
check_test = pd.read_parquet("mia_non_members_test.parquet")

print(f"Members Dataset (Train): {check_train.shape[0]} rows")
print(f"Non-Members Dataset (Test): {check_test.shape[0]} rows")
print("Ready for your Membership Inference Attack!")

--- Fetching 100 Train Rows (Members) ---
Saved 100 rows to mia_members_train.parquet

--- Fetching 100 Test Rows (Non-Members) ---
Saved 100 rows to mia_non_members_test.parquet

--- Final Verification ---
Members Dataset (Train): 100 rows
Non-Members Dataset (Test): 100 rows
Ready for your Membership Inference Attack!


In [1]:
df = pd.read_parquet("../dataset/members.parquet")

NameError: name 'pd' is not defined

In [18]:
df.head()

,text,meta
0,PCI Alternative Using Sustained Exercise (PAUS...,{'pile_set_name': 'PubMed Abstracts'}
1,TiO2 nanotubes for bone regeneration.\nNanostr...,{'pile_set_name': 'PubMed Abstracts'}
2,Standardised protocol for primate faecal analy...,{'pile_set_name': 'PubMed Abstracts'}
3,Examination of factors affecting gait properti...,{'pile_set_name': 'PubMed Abstracts'}
4,Formulation and application of a biosurfactant...,{'pile_set_name': 'PubMed Abstracts'}


In [ ]:
# backup of function to load test data 

# def stream_test_set(max_rows, domain):
#     url = "https://huggingface.co/datasets/mit-han-lab/pile-val-backup/resolve/main/val.jsonl.zst"
#     response = requests.get(url, stream=True)
#     response.raise_for_status()
    
#     dctx = zstd.ZstdDecompressor()
#     stream_reader = dctx.stream_reader(response.raw)
#     text_stream = io.TextIOWrapper(stream_reader, encoding='utf-8')
    
#     extracted_data = []
    
#     for line in text_stream:
#         row = json.loads(line)
#         if row.get("meta", {}).get("pile_set_name") == domain:
#             extracted_data.append(row)
#             if len(extracted_data) >= max_rows:
#                 break
                
#     return extracted_data


# # 1. Stream and extract
# test_non_members_list = stream_test_set(TEST_ROWS_NEEDED, DOMAIN)
